# apc — probe calibration

Robot carries the **70 mm vertical probe rod** (`probe_vertical_70`) — the only tool used here. Recipes load from the project's own **`recipes.j2`**, so each recipe's `calibration_name` matches production.

Calibrate **per clb anchor**: the section near the bottom has a cell for every `clb_` anchor of every component (5 disc adapters + anode), so you can run them one at a time. Each writes one raw/corrected point under the recipe's `calibration_name` in **`core.json`** in this project folder.

After calibrating an anchor you can **test it** — hover above it **with calibration applied** (`recipe.above`), exactly as production would, so you can eyeball that the rod lines up with the real anchor.

In [ ]:
import os, sys, importlib, pkgutil
from pathlib import Path

# ── Project folder ──────────────────────────────────────────────
# core.json is written to Path.cwd(); chdir into the apc project so it lands
# here — the same folder the orchestrator runs main.py from in production.
#
# Self-locating: use this notebook's own directory rather than a hard-coded
# path, so the notebook keeps working if the project folder moves. Jupyter
# stores the kernel's start dir in _dh[0]; fall back to the cwd otherwise.
try:
    _HERE = Path(_dh[0]).resolve()          # Jupyter: notebook's directory
except NameError:
    _HERE = Path.cwd().resolve()            # plain interpreter fallback
assert (_HERE / "recipes.j2").is_file(), (
    f"run this notebook from the apc project folder; _HERE={_HERE} has no recipes.j2"
)
os.chdir(_HERE)
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

# Register component types before the scene loads (library + project-local).
import workspace.components  # noqa: F401
comp_dir = _HERE / "components"
if comp_dir.is_dir():
    for m in pkgutil.iter_modules([str(comp_dir)]):
        if not m.name.startswith("_"):
            importlib.import_module(f"components.{m.name}")

from workspace.workspace import Workspace
from workspace.bt.launcher import load_recipes
print("apc project:", _HERE)

In [ ]:
# Calibration scene = the apc chassis + the calibration layout (production
# layout with the robot carrying probe_vertical_70). Same two-file
# split main.py uses.
SCENE = ["scene/core_500.j2", "scene/calibration.j2"]
ws = Workspace(config_path=[str(_HERE / p) for p in SCENE], port=5000)
core = ws.components["core"]
tool = core.current_tool()
print("probe tool on robot:", None if tool is None else f"{tool.name} ({tool.type})")
assert tool is not None and tool.type == "probe_vertical_70", \
    "expected probe_vertical_70 mounted on the tool changer"

## Load the recipes

Same `recipes.j2` `main.py` uses. We calibrate the 5 disc adapters + the anode.

In [ ]:
rcp = load_recipes(ws, core, _HERE / "recipes.j2")

# The components to calibrate: 2 IN + 3 OUT disc adapters, plus the anode.
CAL_TARGETS = [
    "anode",
    "disc_in_1", "disc_in_2",
    "disc_out_good_1", "disc_out_good_2", "disc_out_bad_1",
]

def clb_anchors(alias):
    """The clb_ anchors on a recipe's component (what calibrate() walks)."""
    return rcp[alias]._resolve_calibration_targets(None)

for alias in CAL_TARGETS:
    r = rcp.get(alias)
    if r is None:
        print(f"{alias:18s} !! not loaded (component missing from scene?)")
        continue
    print(f"{alias:18s} {r.component.name:24s} clb={clb_anchors(alias)}")
    print(f"{'':18s} -> {r.calibration_name}")

## Test one calibrated point

Hover `TEST_CLEARANCE` mm above a chosen anchor **with calibration applied** — this is `recipe.above`, the exact production hover-above motion (it runs the calibration `interpolate` under the hood). After you've calibrated that anchor, run this and eyeball the rod against the physical anchor: if calibration is good, it lines up. Set the component + anchor here.

In [32]:
TEST_ALIAS      = "anode"
TEST_ANCHOR     = "clb_0"
TEST_CLEARANCE  = 30

# Hover the probe rod TEST_CLEARANCE mm above the anchor, WITH calibration
# applied, to eyeball the correction. We can't use recipe.above() here:
# above() bakes in the gripper "face-down" flip (tool_offset [0,0,z,0,180,0]),
# which asks for a 180°-rotated wrist the rod can't reach at the disc-adapter
# corner. Instead we solve IK to the probe's own tcp with no extra rotation —
# exactly the orientation calibration used to reach this anchor. _solve_ik
# still runs interpolate() (r.calibration is on), so the pose is corrected.
r    = rcp[TEST_ALIAS]
body = r.component.assembly["body"]
tool = core.current_tool()
tsol = tool.assembly[next(iter(tool.assembly))]

# +Z in the anchor frame = along the approach axis (rod descends -Z onto it).
J = r._solve_ik(
    body, TEST_ANCHOR, [0, 0, TEST_CLEARANCE, 0, 0, 0],
    tool_dict={"solid": tsol, "anchor": "tcp", "offset": [0, 0, 0, 0, 0, 0]},
)
r.rt.jmove(joint=J, vel=50, accel=500, jerk=3000)


2.0

## Simulation

Run in sim first to watch the motions. In sim there's no contact sensor, so each touch is faked at the nominal pocket — that fake lives in the `probe_vertical_70` component, not the recipe.

In [ ]:
core.simulation(False)

## Calibrate — one cell per clb anchor

Each component has a title and a cell per `clb_` anchor. Run a cell to probe-calibrate just that anchor; it stores one raw/corrected point under the recipe's `calibration_name`. After an anchor, jump up to **Test one calibrated point** (set to that component + anchor) to verify it.

### anode  (`anode`)

In [29]:
rcp["anode"].calibrate("clb_0")


=== anode_1.clb_0  ->  anode_1_True_150_5_5 ===

  -- clb_0.place1 --
     measured floor: [230.6694, -237.9228, 118.6669]

  -- clb_0.place2 --
     measured floor: [256.0948, -238.0912, 118.7707]

  -- clb_0.place3 (centre hole) --
    probe did not trigger within timeout
     centre XY: [243.2952, -238.0363, 121.558]
  [clb_0] raw       = [243.75, -237.5, 113.24, 0.0, 0.0, 180.0]
  [clb_0] corrected = [243.3297, -238.0583, 113.719, 0.6207, -0.395, -179.9976]


True

In [28]:
rcp["anode"].calibrate("clb_1")


=== anode_1.clb_1  ->  anode_1_True_150_5_5 ===

  -- clb_1.place1 --
     measured floor: [231.2269, -337.9767, 118.1163]

  -- clb_1.place2 --
     measured floor: [255.6049, -338.0076, 117.8972]

  -- clb_1.place3 (centre hole) --
    probe did not trigger within timeout
     centre XY: [243.4582, -338.2053, 120.5461]
  [clb_1] raw       = [243.75, -337.5, 113.24, 0.0, 0.0, 180.0]
  [clb_1] corrected = [243.4379, -338.081, 113.0083, -0.3643, 2.2379, -179.9828]


True

### disc in 1  (`disc_in_1`)

In [ ]:
rcp["disc_in_1"].calibrate("clb_0")

In [ ]:
rcp["disc_in_1"].calibrate("clb_1")

### disc in 2  (`disc_in_2`)

In [ ]:
rcp["disc_in_2"].calibrate("clb_0")

In [ ]:
rcp["disc_in_2"].calibrate("clb_1")

### disc out — good 1  (`disc_out_good_1`)

In [ ]:
rcp["disc_out_good_1"].calibrate("clb_0")

In [ ]:
rcp["disc_out_good_1"].calibrate("clb_1")

### disc out — good 2  (`disc_out_good_2`)

In [ ]:
rcp["disc_out_good_2"].calibrate("clb_0")

In [ ]:
rcp["disc_out_good_2"].calibrate("clb_1")

### disc out — bad 1  (`disc_out_bad_1`)

In [ ]:
rcp["disc_out_bad_1"].calibrate("clb_0")

In [ ]:
rcp["disc_out_bad_1"].calibrate("clb_1")

## Stored points

Inspect what landed in `core.json` for a target.

In [ ]:
ALIAS = "anode"
name = rcp[ALIAS].calibration_name
entries = core.calibration.calibration_data.get(name, [])
print(f"{len(entries)} entries under '{name}'")
print(f"core.json -> {Path.cwd() / 'core.json'}")
for e in entries:
    print("  raw       =", [round(v, 3) for v in e["raw"]])
    print("  corrected =", [round(v, 3) for v in e["corrected"]])

## Real robot

Physically mount **probe_vertical_70**, turn simulation off, confirm the probe input reads the **safe** value (`0` = not touching, `1` = touching). Then sweep IK to confirm reach, calibrate an anchor, and hover above it to verify.

In [ ]:
# core.simulation(False)
# assert not core._simulation_mode, "core still in simulation"
# print("probe input 7:", core.dorna.input(index=7), "(expect safe = 0)")
#
# rcp["anode"].calibrate("clb_0")                   # probe one anchor for real
# rcp["anode"].above(anchor="clb_0", padding=15)    # hover above to verify